In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
!nvidia-smi

In [ ]:
import os
WORKDIR = "/kaggle/working/AI_Lab"
os.makedirs(WORKDIR, exist_ok=True)
os.chdir(WORKDIR)
print("Working directory:", os.getcwd())

In [ ]:
import torch
print(torch.__version__, torch.cuda.is_available(), torch.cuda.get_device_name(0))


In [ ]:
!nvcc --version


In [ ]:
!nvcc --version

## Tunnel launch

`TOKEN` and the ngrok authtoken below are placeholders substituted with the real values at push time (see `wake_kaggle()` in `src/dashboard/app.py`) -- nothing sensitive is committed to this notebook file.

In [ ]:
!pip install -q jupyter_http_over_ws pyngrok
!jupyter serverextension enable --py jupyter_http_over_ws


In [ ]:
import subprocess, time
from pyngrok import ngrok

# These two placeholders are substituted with the real values at push time
# by wake_kaggle() in the dashboard app -- never committed as real secrets.
TOKEN = "__JUPYTER_TOKEN_PLACEHOLDER__"
ngrok.set_auth_token("__NGROK_AUTHTOKEN_PLACEHOLDER__")

PORT = 8890
subprocess.Popen([
    "jupyter", "notebook",
    "--NotebookApp.allow_origin=*",
    "--ip=0.0.0.0",
    f"--port={PORT}",
    "--NotebookApp.port_retries=0",
    "--no-browser",
    "--allow-root",
    f"--NotebookApp.token={TOKEN}",
    f"--notebook-dir={WORKDIR}",
])
time.sleep(5)
print("Jupyter server launched on port", PORT, "rooted at", WORKDIR)


In [ ]:
public_url = ngrok.connect(PORT, "http")
print(public_url)
print(TOKEN)


## Keep-alive (activity-based, not a flat timer)

Every real dashboard action (a page-load status check, a Bond message) touches `last_activity.txt` in `WORKDIR` via the same kernel-execution bridge the dashboard already uses. This loop just watches that file and lets the run end on its own once nobody's actually used it for a while — same as closing VS Code/Jupyter and letting a session idle out, not an artificial multi-hour reservation.

In [ ]:
import time, os

ACTIVITY_FILE = os.path.join(WORKDIR, "last_activity.txt")
with open(ACTIVITY_FILE, "w") as f:
    f.write(str(time.time()))

IDLE_TIMEOUT_SECONDS = 20 * 60   # end the run after 20 min with no dashboard activity
CHECK_INTERVAL_SECONDS = 30

print(f"Holding session open -- will end automatically after "
      f"{IDLE_TIMEOUT_SECONDS // 60} min with no dashboard activity.")
while True:
    time.sleep(CHECK_INTERVAL_SECONDS)
    try:
        with open(ACTIVITY_FILE) as f:
            last = float(f.read().strip())
    except Exception:
        last = 0
    idle_for = time.time() - last
    if idle_for > IDLE_TIMEOUT_SECONDS:
        print(f"Idle for {idle_for:.0f}s (> {IDLE_TIMEOUT_SECONDS}s) -- ending session.")
        break
    print(f"Still active ({idle_for:.0f}s since last dashboard activity)")
